# Trích xuất schema cho các dataset đã xử lý

Notebook quét các thư mục dataset trong `data/processed`, đọc tất cả file JSONL (các split), gom `event_type` và `argument_roles`, rồi lưu `schema.json` ngay trong thư mục dataset. Schema được tạo bằng `Schema` và `EventSchema` của dự án.

Chạy lần lượt các cell bên dưới. Nếu `schema.json` đã tồn tại, cell ghi kết quả sẽ thay thế file đó.

In [1]:
import json
import sys
from dataclasses import asdict
from pathlib import Path

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "data" / "processed").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục dự án chứa pyproject.toml và data/processed."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.unified_format.event_schema import EventSchema
from src.unified_format.schema import Schema

DATA_ROOT = PROJECT_ROOT / "data" / "processed"
print(f"Project: {PROJECT_ROOT}")
print(f"Dataset root: {DATA_ROOT}")

Project: d:\Event-Extraction
Dataset root: d:\Event-Extraction\data\processed


In [2]:
def collect_dataset_schema(dataset_dir: Path) -> tuple[Schema, int, int, list[Path]]:
    jsonl_files = sorted(dataset_dir.rglob("*.jsonl"))
    if not jsonl_files:
        raise FileNotFoundError(f"Không tìm thấy file JSONL trong {dataset_dir}")

    roles_by_event: dict[str, set[str]] = {}
    record_count = 0

    for jsonl_path in jsonl_files:
        with jsonl_path.open(encoding="utf-8") as file:
            for line_number, line in enumerate(file, start=1):
                if not line.strip():
                    continue

                try:
                    record = json.loads(line)
                except json.JSONDecodeError as error:
                    raise ValueError(
                        f"JSON không hợp lệ tại {jsonl_path}:{line_number}"
                    ) from error

                if not isinstance(record, dict):
                    raise ValueError(
                        f"Record phải là object tại {jsonl_path}:{line_number}"
                    )
                events = record.get("events")
                if not isinstance(events, list):
                    raise ValueError(
                        f"Trường 'events' phải là list tại {jsonl_path}:{line_number}"
                    )

                record_count += 1
                for event_index, event in enumerate(events):
                    location = f"{jsonl_path}:{line_number}, event {event_index}"
                    if not isinstance(event, dict):
                        raise ValueError(f"Event phải là object tại {location}")

                    event_type = event.get("event_type")
                    if not isinstance(event_type, str) or not event_type.strip():
                        raise ValueError(
                            f"Thiếu 'event_type' hợp lệ tại {location}"
                        )
                    arguments = event.get("arguments")
                    if not isinstance(arguments, list):
                        raise ValueError(
                            f"Trường 'arguments' phải là list tại {location}"
                        )

                    event_roles = roles_by_event.setdefault(event_type, set())
                    for argument_index, argument in enumerate(arguments):
                        argument_location = f"{location}, argument {argument_index}"
                        if not isinstance(argument, dict):
                            raise ValueError(
                                f"Argument phải là object tại {argument_location}"
                            )
                        role = argument.get("role")
                        if not isinstance(role, str) or not role.strip():
                            raise ValueError(
                                f"Thiếu 'role' hợp lệ tại {argument_location}"
                            )
                        event_roles.add(role)

    schema = Schema(
        event_schemas=[
            EventSchema(
                event_type=event_type,
                argument_roles=sorted(roles_by_event[event_type]),
            )
            for event_type in sorted(roles_by_event)
        ]
    )
    return schema, len(jsonl_files), record_count, jsonl_files

In [3]:
dataset_dirs = sorted(
    path
    for path in DATA_ROOT.iterdir()
    if path.is_dir() and any(path.rglob("*.jsonl"))
)
if not dataset_dirs:
    raise FileNotFoundError(f"Không tìm thấy dataset JSONL trong {DATA_ROOT}")

summary = []
for dataset_dir in dataset_dirs:
    schema, file_count, record_count, _ = collect_dataset_schema(dataset_dir)
    output_path = dataset_dir / "schema.json"
    output_path.write_text(
        json.dumps(asdict(schema), ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    summary.append(
        {
            "dataset": dataset_dir.name,
            "jsonl_files": file_count,
            "records": record_count,
            "event_types": len(schema.event_schemas),
            "schema_path": str(output_path),
        }
    )

print("| Dataset | JSONL files | Records | Event types | Schema output |")
print("|---|---:|---:|---:|---|")
for row in summary:
    print(
        f"| {row['dataset']} | {row['jsonl_files']} | {row['records']} "
        f"| {row['event_types']} | `{row['schema_path']}` |"
    )

| Dataset | JSONL files | Records | Event types | Schema output |
|---|---:|---:|---:|---|
| BKEE | 3 | 17108 | 33 | `d:\Event-Extraction\data\processed\BKEE\schema.json` |
| GENEVA | 3 | 3570 | 115 | `d:\Event-Extraction\data\processed\GENEVA\schema.json` |
| GENEVAv1 | 3 | 3684 | 115 | `d:\Event-Extraction\data\processed\GENEVAv1\schema.json` |
| PHEE | 3 | 4763 | 2 | `d:\Event-Extraction\data\processed\PHEE\schema.json` |
| VHE | 3 | 4070 | 33 | `d:\Event-Extraction\data\processed\VHE\schema.json` |
